# RAG - Full Implementation with LLM

I am gonna build these to make a full flow.

1. Write a simple document loader.
2. Implement chunking.
3. Generate embeddings.
4. Build a vector store.
5. Retrieve the top-k chunks.
6. Add a reranker.
7. Construct the prompt.
8. Send it to an LLM.
9. Evaluate the results.

**Prompt construction**:

```
You are a helpful assistant.

Answer only using the supplied context.

Context:

--------------------

Chunk 12

...

--------------------

Chunk 18

...

--------------------

Chunk 31

...

Question:

How does Product Quantisation work?
```

If the prompt says:

> If the answer cannot be found in the supplied context,
> say you don't know.

the model is much less likely to hallucinate.

**Context Management**

Not all chunks are useful. So common approach is:

1. Retrieve the top k chunks
2. Deduplicate nearly identical chunks
3. Discard irrelevant ones
4. Keep only the best few that fit in the prompt

**Reranking**

Say the retriever returns top k, the reranker filters most relevant k' ones (k' < k)

**Evaluation**

Model evaluation

Retrieval metrics:

- Recall@k: Does the correct chunk appear in the top k results?
- Precision@k: Of the retrieved chunks, how many are actually relevant?
- Mean Reciprocal Rank (MRR): How high is the first relevant chunk ranked?
- nDCG: Rewards placing highly relevant documents near the top while accounting for graded relevance.

Generation metrics:

- Care about the final answer
- Many teams use a mixture of human evaluation, benchmark question–answer datasets, and increasingly, LLM-as-a-judge—while being mindful that automated judges have their own limitations.

**Putting it all together**

Documents > Chunking > Embeddings > Vector DB > ANN Search (FAISS) > Top-k Chunks > Reranker > Selected Context > Prompt Construction > LLM > Answer > Evaludation.

## Init

In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

/Users/dunghq/code/github/ai/having_fun/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configs

In [2]:
# all-MiniLM-L6-v2 is small, fast, and free. Perfect for learning
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

CHUNK_SIZE = 300
CHUNK_OVERLAP = 50

TOP_K = 5

## Flow

### Document Loader

In [3]:
@dataclass
class Document:
    id: int
    text: str
    source: str

In [4]:
def load_documents(folder: str):
    '''
    USAGE: documents = load_documents("data")
    '''
    documents = []

    for i, path in enumerate(Path(folder).glob("*.txt")):
        text = path.read_text(encoding="utf-8")
        documents.append(Document(
            id=i,
            text = text,
            source=path
        ))
    
    return documents

In [5]:
documents = load_documents("data")
print(len(documents))

3


### Chunking

Vector DB doesn't store documents, it stores chunks.

In [6]:
@dataclass
class Chunk:
    id: int
    document_id: int
    text: str

In [7]:
def chunk_text(text: str, chunk_size: int, overlap: int):
    '''
    Separate text to chunks
    '''
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    
    return chunks

In [8]:
def create_chunks(documents: list[Document]):
    '''
    Convert all documents to chunks.

    USAGE: chunks = create_chunks(documents)
    '''
    all_chunks = []
    chunk_id = 0

    for doc in documents:
        texts = chunk_text(
            doc.text,
            CHUNK_SIZE,
            CHUNK_OVERLAP
        )

        for t in texts:
            all_chunks.append(Chunk(
                id=chunk_id,
                document_id=doc.id,
                text=t
            ))
            chunk_id += 1
    
    return all_chunks

In [9]:
# Chunking
chunks = create_chunks(documents)
print(len(chunks))

12


### Embedding

In [10]:
# Load model
model = SentenceTransformer(EMBEDDING_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15725.59it/s]


In [11]:
# Embed every chunk
texts = [chunk.text for chunk in chunks]

embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.19it/s]


In [12]:
# Put everything together
@dataclass
class EmbeddedChunk:
    chunk: Chunk
    embedding: np.ndarray

In [13]:
embedded_chunks = [
    EmbeddedChunk(chunk, embedding)
    for chunk, embedding in zip(chunks, embeddings)
]

### Build a Vector Store

In [24]:
# If using FAISS, we only need index.search(query_vector).
# Cosine is no longer needed.

def cosine_similarity(query: np.array, matrix: np.ndarray):
    """
    query (d,)
    matrix (n, d)

    returns:
        (n,)
    """
    query_norm = np.linalg.norm(query)
    matrix_norm = np.linalg.norm(matrix, axis=1)

    return (matrix @ query) / (matrix_norm * query_norm)

In [15]:
@dataclass
class SearchResult:
    chunk: Chunk
    score: float

In [16]:
class VectorStore:

    def __init__(self):
        self.embeddings = None
        self.chunks = []

    def add(self, embedded_chunk: EmbeddedChunk):
        if self.embeddings is None:
            self.embeddings = embedded_chunk.embedding.reshape(1, -1)
        else:
            self.embeddings = np.vstack([
                self.embeddings,
                embedded_chunk.embedding
            ])
        self.chunks.append(embedded_chunk.chunk)
    
    def search(self, query: np.array, model, top_k=5):
        query_embedding = model.encode(query, convert_to_numpy=True)
        scores = cosine_similarity(query_embedding, self.embeddings)
        indices = np.argsort(scores)[::-1][:top_k]
        results = []

        for idx in indices:
            results.append(SearchResult(
                chunk = self.chunks[idx],
                score = scores[idx]
            ))
        
        return results

In [18]:
# Build it
vector_store = VectorStore()

for item in embedded_chunks:
    vector_store.add(item)

### Retrieval

In [19]:
def embed_query(model, query):
    return model.encode(query, convert_to_numpy=True)

In [20]:
# Test
query_embedding = embed_query(
    model,
    "How does FAISS work?"
)

### Find Top-K

In [21]:
def top_k(scores: np.ndarray, k: int):
    indices = np.argsort(scores)[::-1][:k]
    return indices

### Search

A method `search` is added into `VectorStore` class.

## Test

In [23]:
results = vector_store.search(
    "How does FAISS work?",
    model, top_k=3
)

for r in results:
    print("="*80)
    print(f"Score: {r.score:.3f}")
    print(r.chunk.text)

Score: 0.252
Facebook AI Similarity Search (FAISS) is an open-source library for dense vectors.
It allows developers to search for multimedia documents that are similar.
Traditional databases are not optimized for searching high-dimensional vector embeddings.
FAISS excels at searching these embeddings at scale w
Score: 0.246
arch implementations for L2 distance and inner product.
Some of its most efficient algorithms are implemented to run on GPUs.
FAISS uses indexing methods to drastically accelerate the search process.
It is a foundational tool for building production-grade RAG retrieval pipelines.

Score: 0.201
 most similar context.
The retrieved context is appended to the original user prompt.
This combined prompt is then sent to the language model.
The LLM generates an accurate response grounded in the provided facts.
This process significantly reduces hallucinations and improves response reliability.

